## Installs

In [ ]:
!pip install -U torch torchvision
!pip install transformers datasets pandas tqdm scipy

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy # Only needed within runpod environment
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import ViTForImageClassification
from datasets import load_from_disk, concatenate_datasets
import pandas as pd
import numpy as np
import os
from collections import defaultdict
import copy

## Data Prep

In [ ]:
train = load_from_disk('/workspace/preprocessed/SUN397/train_processed')
val = load_from_disk('/workspace/preprocessed/SUN397/val_processed')
test = load_from_disk('/workspace/preprocessed/SUN397/test_processed')

In [ ]:
train.set_format(type='torch', columns=["image", "label", "pixel_values"])
val.set_format(type='torch', columns=["image", "label", "pixel_values"])
test.set_format(type='torch', columns=["image", "label", "pixel_values"])

def ViT_collate_fn(batch):
    images = torch.stack([example["pixel_values"] for example in batch])
    labels = torch.tensor([example["label"] for example in batch])
    
    return {
        "pixel_values": images,
        "labels": labels
    }

val_loader = DataLoader(val, batch_size=8, shuffle=False, collate_fn=ViT_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
test_loader = DataLoader(test, batch_size=8, shuffle=False, collate_fn=ViT_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## Model Classes Prep

In [ ]:
class Hooks(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.CLS = []
    
    def forward(self, images):
        self.CLS = []
        
        hidden_states = self.model.vit.embeddings(images)

        for i, layer_module in enumerate(self.model.vit.encoder.layer):
            layer_outputs = layer_module(hidden_states)
            hidden_states = layer_outputs[0]
            self.CLS.append(hidden_states[:, 0, :])

        return self.CLS    

In [ ]:
# https://github.com/huggingface/transformers/blob/main/src/transformers/models/vit/modeling_vit.py 
class Augmented(torch.nn.Module):
    def __init__(self, model, classifier=None, W=None, transform_stage=-1):
        super().__init__()
        self.model = model
        self.classifier = classifier if classifier is not None else torch.nn.Linear(self.model.config.hidden_size, 397)
        self.W = torch.from_numpy(W.astype(np.float32)).to(device) if W is not None else None
        self.transform_stage = transform_stage

    def forward(self, image):
        hidden_states = self.model.vit.embeddings(image)

        for i, layer_module in enumerate(self.model.vit.encoder.layer):
            layer_outputs = layer_module(hidden_states)
            hidden_states = layer_outputs[0]
            if self.transform_stage == i:
                if self.W is None:
                    self.W = torch.eye(hidden_states.shape[-1], device=hidden_states.device, dtype=hidden_states.dtype)
                cls = hidden_states[:, 0, :]
                cls = cls @ self.W
                hidden_states[:,0,:] = cls
                break
            cls = hidden_states[:, 0, :]

        sequence_output = self.model.vit.layernorm(hidden_states)
        logits = self.classifier(sequence_output[:, 0, :])

        return logits, cls

In [ ]:
# https://huggingface.co/google/vit-large-patch16-384 
refer = ViTForImageClassification.from_pretrained('google/vit-large-patch16-384').to(device)
f_t = Augmented(copy.deepcopy(refer))
f_t.load_state_dict(torch.load('best_Google_ViT_SUN397.pt'))

base_H = Hooks(copy.deepcopy(refer))
fine_tuned_H = Hooks(f_t.model).to(device)

In [ ]:
base = Augmented(copy.deepcopy(refer))
base = base.eval().to(device)

fine_tuned = Augmented(copy.deepcopy(f_t.model), f_t.classifier)
fine_tuned = fine_tuned.eval().to(device)

## Evaluating Prep

In [ ]:
def cosineSimilarity(fine_tuned_cls, aug_cls):
    eps = 1e-8
    out_aug = F.normalize(aug_cls, dim=1, eps=eps)
    out_fine = F.normalize(fine_tuned_cls, dim=1, eps=eps)

    cos_sim = (out_aug * out_fine).sum(dim=1).mean().item()
    return cos_sim

In [ ]:
# 1/10 of Test Image Set. 2600 total images. 260 images per label max. 
size_nums = [i for i in range(1, 28)] # Start from 1,2,3,4,...->25. 27 total.
size_nums.extend([30, 35, float('inf')]) # add 3. range(30)
train_size = 0

labels = train["label"]

label_to_indices = defaultdict(list)

for idx, label in enumerate(labels):
    label = int(label)
    label_to_indices[label].append(idx)

In [ ]:
filtered_train = {
    label: train.select(indices) for label, indices in label_to_indices.items()
}

for label, ds in filtered_train.items():
    print(f"Label {label}: {len(ds)} examples")

In [ ]:
indices = [i for i in range(0, refer.vit.encoder.config.num_hidden_layers, 2)] + [23]

for s in size_nums:
    train_indice = s

    sorted = []
    for i in range(397):
        num_indices = min(train_indice, len(filtered_train[i]))
        sorted.append(filtered_train[i].select(range(num_indices)))
    
    train_dataset = concatenate_datasets(sorted)
    train_size = len(train_dataset)
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=ViT_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
    print(f"Results for {train_indice} Image(s) per class: ({train_size} Training Images -> 19850 Testing Images)")

    # Extracting
    Z0 = {i: [] for i in indices}
    Z1 = []

    with torch.no_grad():
        for batch in tqdm(train_loader, desc="Extracting"):
            images = batch["pixel_values"].to(device, non_blocking=True)

            out_base = base_H(images)
            out_fine_tuned = fine_tuned_H(images)

            for i in indices:
                Z0[i].append(out_base[i].float().cpu())
            Z1.append(out_fine_tuned[-1].float().cpu())

    Z1_last = torch.cat(Z1)
    Z1 = Z1_last.cpu().numpy()

    W = {}
    resid = {}

    for i, val in Z0.items():
        val = torch.cat(val)
        val = val.cpu().numpy()
        W[i], resid[i], _, _ = np.linalg.lstsq(val, Z1, rcond=None)

    # Augmenting Models
    aug = {}

    for i in indices:
        model = Augmented(copy.deepcopy(refer), classifier=f_t.classifier, W=W[i], transform_stage=i)
        model = model.eval().to(device)
        aug[i] = model

    # Evaluating Augmented Models
    correct_base = 0
    correct_fine_tuned = 0
    total_samples = 0
    
    correct = {i: 0 for i in indices}
    co_sim_cls = {i: [] for i in indices}

    for batch in tqdm(test_loader, desc="Evaluating"):
        images = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        total_samples += labels.size(0)

        # Base Model
        logits_base, cls_base = base(images)
        predicted = logits_base.argmax(dim=1)
        correct_base += (predicted == labels).sum().item()

        # Fine-Tuned Model
        logits_fine_tuned, cls_fine_tuned = fine_tuned(images)
        predicted = logits_fine_tuned.argmax(dim=1)
        correct_fine_tuned += (predicted == labels).sum().item()

        # Augmented Models
        for i in indices:
            logits_aug, cls_aug = aug[i](images)
            predicted = logits_aug.argmax(dim=1)
            correct[i] += (predicted == labels).sum().item()
            co_sim_cls[i].append(cosineSimilarity(cls_fine_tuned, cls_aug))

    base_acc = correct_base / total_samples
    fine_tuned_acc = correct_fine_tuned / total_samples

    for i in indices:
        correct[i] = correct[i] / total_samples
        co_sim_cls[i] = np.mean(co_sim_cls[i])
    
    print(f"Augmented Google ViT on SUN397 Results")
    for i in indices:
        print(f"\tAugmented {i} - Last ({indices[-1]}) Layer Accuracy: {correct[i]}")
        print(f"\tAverage Cosine Similarity of CLS Token of Augmented {i} Layer: {co_sim_cls[i]:.4f}")
    print(f"Base Accuracy: {base_acc:.4f}")
    print(f"Fine-Tuned Accuracy: {fine_tuned_acc:.4f}")

    folder = "./Results/SUN397/Entire_Transformation_Matrix_W"
    os.makedirs(folder, exist_ok=True)

    data = {
        'Images_Per_Label': [train_indice] * len(indices),
        'Train_Data_Size': [train_size] * len(indices),
        'W': [W[i] for i in indices],
        'Residuals': [resid[i] for i in indices],
        'Classification_Accuracy': [correct[i] for i in indices],
        'CLS_Cosine_Similarity': [co_sim_cls[i] for i in indices],
    }

    df = pd.DataFrame(data, index=indices)
    name = f"Transformation_Matrix_W_{train_size}_Augmentation_Results.csv"
    path = os.path.join(folder, name)
    df.to_csv(path)

print("Ablation Results")

aug = {}

for i in indices:
    model = Augmented(copy.deepcopy(refer), classifier=f_t.classifier, transform_stage=i)
    model = model.eval().to(device)
    aug[i] = model

# Evaluating Augmented Models
correct_base = 0
correct_fine_tuned = 0
total_samples = 0

correct = {i: 0 for i in indices}
co_sim_cls = {i: [] for i in indices}

for batch in tqdm(test_loader, desc="Evaluating"):
    images = batch["pixel_values"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)
    total_samples += labels.size(0)

    # Base Model
    logits_base, cls_base = base(images)
    predicted = logits_base.argmax(dim=1)
    correct_base += (predicted == labels).sum().item()

    # Fine-Tuned Model
    logits_fine_tuned, cls_fine_tuned = fine_tuned(images)
    predicted = logits_fine_tuned.argmax(dim=1)
    correct_fine_tuned += (predicted == labels).sum().item()

    # Augmented Models
    for i in indices:
        logits_aug, cls_aug = aug[i](images)
        predicted = logits_aug.argmax(dim=1)
        correct[i] += (predicted == labels).sum().item()
        co_sim_cls[i].append(cosineSimilarity(cls_fine_tuned, cls_aug))

base_acc = correct_base / total_samples
fine_tuned_acc = correct_fine_tuned / total_samples

for i in indices:
    correct[i] = correct[i] / total_samples
    co_sim_cls[i] = np.mean(co_sim_cls[i])

print(f"Augmented Google ViT on SUN397 Results")
for i in indices:
    print(f"\tAugmented {i} - Last ({indices[-1]}) Layer Accuracy: {correct[i]}")
    print(f"\tAverage Cosine Similarity of CLS Token of Augmented {i} Layer: {co_sim_cls[i]:.4f}")
print(f"Base Accuracy: {base_acc:.4f}")
print(f"Fine-Tuned Accuracy: {fine_tuned_acc:.4f}")

folder = "./Results/SUN397/Entire_Transformation_Matrix_W"
os.makedirs(folder, exist_ok=True)

data = {
    'Classification_Accuracy': [correct[i] for i in indices],
    'CLS_Cosine_Similarity': [co_sim_cls[i] for i in indices],
}

df = pd.DataFrame(data, index=indices)
name = f"Ablation_Results.csv"
path = os.path.join(folder, name)
df.to_csv(path)

f_t = Augmented(copy.deepcopy(refer))
f_t.load_state_dict(torch.load("best_Google_ViT_SUN397_Linear_Probe.pt", map_location=device))

print("Linear Probe Results")

aug = {}

for i in indices:
    model = Augmented(copy.deepcopy(refer), classifier=f_t.classifier, transform_stage=i)
    model = model.eval().to(device)
    aug[i] = model

# Evaluating Augmented Models
correct_base = 0
correct_fine_tuned = 0
total_samples = 0

correct = {i: 0 for i in indices}
co_sim_cls = {i: [] for i in indices}

for batch in tqdm(test_loader, desc="Evaluating"):
    images = batch["pixel_values"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)
    total_samples += labels.size(0)

    # Base Model
    logits_base, cls_base = base(images)
    predicted = logits_base.argmax(dim=1)
    correct_base += (predicted == labels).sum().item()

    # Fine-Tuned Model
    logits_fine_tuned, cls_fine_tuned = fine_tuned(images)
    predicted = logits_fine_tuned.argmax(dim=1)
    correct_fine_tuned += (predicted == labels).sum().item()

    # Augmented Models
    for i in indices:
        logits_aug, cls_aug = aug[i](images)
        predicted = logits_aug.argmax(dim=1)
        correct[i] += (predicted == labels).sum().item()
        co_sim_cls[i].append(cosineSimilarity(cls_fine_tuned, cls_aug))

base_acc = correct_base / total_samples
fine_tuned_acc = correct_fine_tuned / total_samples

for i in indices:
    correct[i] = correct[i] / total_samples
    co_sim_cls[i] = np.mean(co_sim_cls[i])

print(f"Augmented Google ViT on SUN397 Results")
for i in indices:
    print(f"\tAugmented {i} - Last ({indices[-1]}) Layer Accuracy: {correct[i]}")
    print(f"\tAverage Cosine Similarity of CLS Token of Augmented {i} Layer: {co_sim_cls[i]:.4f}")
print(f"Base Accuracy: {base_acc:.4f}")
print(f"Fine-Tuned Accuracy: {fine_tuned_acc:.4f}")

folder = "./Results/SUN397/Entire_Transformation_Matrix_W"
os.makedirs(folder, exist_ok=True)

data = {
    'Classification_Accuracy': [correct[i] for i in indices],
    'CLS_Cosine_Similarity': [co_sim_cls[i] for i in indices],
}

df = pd.DataFrame(data, index=indices)
name = f"Linear_Probe_Augmentation_Results.csv"
path = os.path.join(folder, name)
df.to_csv(path)